# Notebook 04 — Model Improvement

Objective: class weighting (Experiment B), optional SMOTE (Experiment C), hyperparameter tuning, and threshold analysis.

Model selection must be based on evidence from these experiments — not assumed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from config import (
    METRICS_DIR, FIGURES_DIR,
    LR_PARAM_GRID, DT_PARAM_GRID, RF_PARAM_GRID,
    TUNING_SCORER,
)
from src.data_processing import load_splits
from src.feature_engineering import (
    build_lr_pipeline, build_dt_pipeline, build_rf_pipeline,
    build_lr_smote_pipeline, build_rf_smote_pipeline,
)
from src.train import run_baseline, cross_validate_model, tune_model
from src.evaluate import (
    compute_metrics, print_metrics_table,
    plot_confusion_matrix, plot_roc_curves, plot_pr_curves,
    threshold_analysis, save_metrics_csv,
)

METRICS_DIR.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

In [ ]:
X_train, X_test, y_train, y_test = load_splits()
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

## Experiment B — class_weight='balanced'

Compare models trained with balanced class weighting against the Experiment A baselines.

In [ ]:
pipe_lr_b = build_lr_pipeline(class_weight='balanced')
pipe_dt_b = build_dt_pipeline(class_weight='balanced')
pipe_rf_b = build_rf_pipeline(class_weight='balanced')

print('LR balanced CV:')
cross_validate_model(pipe_lr_b, X_train, y_train)

In [ ]:
print('DT balanced CV:')
cross_validate_model(pipe_dt_b, X_train, y_train)

In [ ]:
print('RF balanced CV:')
cross_validate_model(pipe_rf_b, X_train, y_train)

## Experiment C (Optional) — SMOTE

SMOTE is fitted inside the imblearn Pipeline so it is applied per-fold during CV and never touches the test set.

In [ ]:
# Uncomment to run SMOTE experiment
# pipe_lr_smote = build_lr_smote_pipeline()
# cross_validate_model(pipe_lr_smote, X_train, y_train)

# pipe_rf_smote = build_rf_smote_pipeline()
# cross_validate_model(pipe_rf_smote, X_train, y_train)
print('SMOTE cells commented — uncomment to run.')

## Hyperparameter Tuning

Optimise for PR-AUC (average_precision) — justified by severe class imbalance.

Tune on training data only. Test set is NOT involved.

In [ ]:
print(f'Tuning scorer: {TUNING_SCORER}')
print(f'LR grid size: {len(LR_PARAM_GRID)} param sets')

In [ ]:
best_lr, best_lr_params, lr_cv_results = tune_model(
    build_lr_pipeline(), LR_PARAM_GRID, X_train, y_train
)

In [ ]:
best_dt, best_dt_params, dt_cv_results = tune_model(
    build_dt_pipeline(), DT_PARAM_GRID, X_train, y_train
)

In [ ]:
best_rf, best_rf_params, rf_cv_results = tune_model(
    build_rf_pipeline(), RF_PARAM_GRID, X_train, y_train
)

## Tuned Model Comparison

In [ ]:
# Evaluate tuned models on the test set
y_pred_lr_t,  y_proba_lr_t  = run_baseline(best_lr, X_train, y_train, X_test)
y_pred_dt_t,  y_proba_dt_t  = run_baseline(best_dt, X_train, y_train, X_test)
y_pred_rf_t,  y_proba_rf_t  = run_baseline(best_rf, X_train, y_train, X_test)

m_lr_t = compute_metrics(y_test, y_pred_lr_t, y_proba_lr_t, 'LR Tuned')
m_dt_t = compute_metrics(y_test, y_pred_dt_t, y_proba_dt_t, 'DT Tuned')
m_rf_t = compute_metrics(y_test, y_pred_rf_t, y_proba_rf_t, 'RF Tuned')

tuned_comparison = print_metrics_table([m_lr_t, m_dt_t, m_rf_t])
save_metrics_csv(tuned_comparison, 'tuned_results.csv')

## Threshold Analysis

For the strongest candidate model (fill in after examining comparison table).

In [ ]:
# Change the model and probabilities below to match your chosen candidate
CANDIDATE_MODEL  = best_rf       # <-- update if needed
CANDIDATE_PROBA  = y_proba_rf_t  # <-- update if needed
CANDIDATE_LABEL  = 'Random Forest (tuned)'  # <-- update

thresh_df = threshold_analysis(y_test, CANDIDATE_PROBA, CANDIDATE_LABEL)
save_metrics_csv(thresh_df.set_index('threshold'), 'threshold_results.csv')
thresh_df.head(10)

## Final Model Selection

Document the chosen model and threshold here based on experimental evidence.

| Criterion | Decision |
|---|---|
| Selected model | (fill in) |
| Justification | (recall, precision, F1, PR-AUC) |
| Threshold | (fill in) |
| Threshold rationale | (e.g., recall/precision trade-off) |

Proceed to Notebook 05 to apply SHAP and save the final model.